In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from pathlib import Path
from bs4 import BeautifulSoup

In [ ]:
#In this case, the coding depends on a version of the KB BERT model trained for Sentiment Analysis.
tokenizer = AutoTokenizer.from_pretrained("KBLab/megatron-bert-large-swedish-cased-165k")
model = AutoModelForSequenceClassification.from_pretrained("KBLab/robust-swedish-sentiment-multiclass")
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

#I also establish the timespan relevant for the study in question.
timespan = [x for x in range(1887, 1915)]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/904k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/906 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

In [ ]:
class Statement:
    """
    Takes a sentence and stores its sentiment and other relevant information.
    
    Attributes:
    statement (str): The statement to be analyzed.
    sentiment (dict): The result of the sentiment analysis.
    label (str): The label returned by the SA (either POSITIVE, NEUTRAL or NEGATIVE)
    score (float): A value between 0 and 1 designating how positive the statemnt was perceived.
    """
    def __init__(self, statement):
        self.statement = statement
        self.sentiment = {}
        self.label = ""
        self.score = 0.0

        #SA returns a list of one dict, which is here added as the self.sentiment.
        self.sentiment = classifier(self.statement)[0]
        for key, val in self.sentiment.items():
          if key == 'label':
            self.label = val
          if key == "score":
            self.score = val
        pass

    def __str__(self):
      return f"Label: {self.label} Score: {self.score}"
      pass


In [ ]:
#My project looks at how certain locations have figured in parliamentary debates, so the following class is designed to identify the
#sentiment of the context in which the location occurs.
class Speech:
  """
  Reads a file, identifies the sentences where a certain location is mentioned, and stores their sentiment analyses.
  
  Attributes:
  path (str): The path of the file.
  loc (str): The location that is to be analysed.
  year (str): The year the speech was held.
  statements (list[Statement]): A list of Statements for each sentence where the location is mentioned.
  """
  def __init__(self, path, loc):
    self.path = path
    self.loc = loc
    self.year = ""
    self.statements = []

    #The year is extracted from the file name.
    stringpath = str(self.path)
    split_path = stringpath.split("prot-")
    try:
      half_path = split_path[1]
      self.year = half_path[:4]
    #One file in every folder contains metadata and is sorted out.
    except IndexError:
      self.year = "metadata"

    #The content of the file is read and split into sentences.
    with open (self.path, "r", encoding="utf8") as fs:
      full_speech = fs.read()
    sentence_list = full_speech.split(".")

    #The sentences are then processed with SA.
    for sentence in sentence_list:
      if self.loc in sentence:
        sentiment_findings = Statement(sentence)
        self.statements.append(sentiment_findings)
    pass

In [ ]:
class Sum:
  """
  Analyses a directory of parliamentary speeches and stores the SA information in .txt file.
  
  Attributes:
  directory (str): The path of the directory.
  loc (str): The location that is to be analysed.
  statements (list[list[str, Statement]]): List of small lists pairing Statements with their year.
  """
  def __init__(self, directory, loc):
    self.dir = directory
    self.loc = loc
    self.statements = []

    #For every file in the directory, the sentences in which the location is mentiones is processed with SA and stored together with 
    #the corresponding year in the self.statements list.
    directory_path = Path(self.dir)
    for file in directory_path.iterdir():
      if file.is_file():
        speech = Speech(file, self.loc)
        for statement in speech.statements:
          statement_note = [speech.year, statement]
          self.statements.append(statement_note)

    pass

#For analysis purposes, the location data for each year is taken together.
  def yearmean(self, year):
    """
    Calculates the mean sentiment of the given location for a certain year.
    
    Args:
    year (str): The year the is to be analyzed.
    
    Returns:
    The mean score and the total number of labels for the given location the given year.
    """
    #The scores and labels of the year are added together.
    score_sum = 0.0
    labellist = []

    for statement in self.statements:
      #The first item of each listed list is the year of the Statement.
      if statement[0] == year:
        #The second item is a Statments class, having both a self.score and self.label that are added to the year totals.
        score_sum += note[1].score
        labellist.append(note[1].label)

    #When the totals have been added together, the total number of the different labels are counted.
    poscount = labellist.count("POSITIVE")
    neucount = labellist.count("NEUTRAL")
    negcount = labellist.count("NEGATIVE")
    
    try:
      #The mean score is calculated by dividing by the number of Statements, which corresponds to the number of labels listed. The result
      #is then rounded for simplicity and returned as a line listing the information. The format is designed to simplify reading the
      #information in the future.
      score_mean = score_sum / len(labellist)
      rounded_mean = round(score_mean, 2)
      return(f"{year}--{rounded_mean}--{poscount}--{neucount}--{negcount}")
    except ZeroDivisionError:
      #If there are no mentions of the location a certain year, the function returns an allert acknowledging that fact.
      stringyear = str(year)
      return("No matches in " + stringyear)

    pass


  def table(self, startyear=1887, endyear=1914):
    """
    Prints the sentiment data year by year.
    
    Args:
    startyear (int): The first year in the range, default set to the beginning of my current project.
    endyear (int): The last year in the range, default set to the end of my current project
    
    Returns:
    Printed sentiment info for a certain location year by year.
    """
    #Python does not include the last number of a range, so the endyear is augmented to work more intuitively.
    endyear += 1
    yearlist = [x for x in range(startyear, endyear)]
    
    #For every year, the function prints the summarised sentiment information.
    for year in yearlist:
      yearstring = str(year)
      print(self.yearmean(yearstring))
    
    pass

  def store(self):
    """
    Stores the sentiment information to .txt file to be analysed later.
    
    Returns:
    .txt file with sentiment info.
    """
    #The compiled sentiment info for every year is stored in a list.
    periodlist = []
    for year in timespan:
      yearstring = str(year)
      periodlist.append(self.yearmean(yearstring))

    #The list is sorted, made into a str and written into the .txt file.
    periodlist.sort()
    total_sentiment = str(periodlist)
    with open(f"/content/drive/MyDrive/Colab Notebooks/Sentiments/{self.loc}.txt", "w") as storefile:
      storefile.write(total_sentiment)

    pass

In [ ]:
#The following classes are designed to handle the stored sentiment data and present it in relevant ways.
class YearData:
  """
  Stores the information of a certain year in the stored .txt file in an object.
  
  Attributes:
  data (str): A string on the template defined in the .yearmean() function.
  """
  def __init__(self, data):
    self.data = data
    self.year = 0
    self.mean = 0.0
    self.pos = 0
    self.neu = 0
    self.neg = 0

    #The strict formatting allows for a simple split and then allocating each list element to its corresponding value.
    dist_data = self.data.split("--")
    self.year = int(dist_data[0])
    self.mean = float(dist_data[1])
    self.pos = int(dist_data[2])
    self.neu = int(dist_data[3])
    selg.neg = int(dist_data[4])
    pass

  def __str__(self):
    return(f"{self.year}: {self.mean} ({self.pos}/{self.neu}/{self.neg})")

  def getYear(obj):
    return obj.year

In [ ]:
class LocationSentiment:
  """
  Summarizes the sentiment data of statements including a particular location over a number of years
  
  Attributes:
  path (str): Path to the file storing the sentiment data.
  data (list[YearData]): Lists the sentiment data of every year.
  """
  def __init__(self, path):
    self.path = path
    self.data = []

    #The stored file is read and its content split into elements corresponding to different years.
    with open(self.path, "r", encoding="utf8") as sf:
      stored_file = sf.read()
      stored_list = stored_file.split("'")
      
      #The sentiment data of each year is then stored as a YearData object.
      for element in stored_list:
        if element[0] == "1":
          info = YearData(element)
          self.data.append(info)

    pass




In [ ]:
class Group:
  """
  Adds together and presents the stored sentiment data of a number of locations.
  
  Attributes:
  dir (str): The path of the directory with the stored .txt files.
  path (Path): A Path object made from the self.path.
  sum (list[YearData]): Holds all the data of all the years of the designated locations.
  """
  def __init__(self, directory):
    self.dir = directory
    self.path = Path(self.dir)
    self.sum = []

    #For each file in the directory, the class creates a LocationSentiment object and adds its sentiment data year by year to self.sum.
    #Each file corresponds to a separate location that will be analysed together with the others.
    for file in self.path.iterdir():
      if file.is_file():
        location_info = LocationSentiment(file)
        self.sum.extend(location_info.data)
        self.sum.sort(key= getYear)

    pass

  def table(self):
    """
    Adds the sentiment data of the various locations together and prints the result in a table.
    
    Returns:
    Prints the findings of the sentiment analysis of n different locations taken together year by year.
    """
    #Starting sums are established for every year and then added to with the info of the corresponding objects.
    for year in timespan:
      mean_sum = 0.0
      pos_sum = 0
      neu_sum = 0
      neg_sum = 0
      data_count = 0
      for data in self.sum:
        if data.year == year:
          mean_sum += data.mean
          pos_sum += data.pos
          neu_sum += data.neu
          neg_sum += data.neg
          data_count += 1
      
      #The mean is calculated and the summarized info printed. In cases where no references have been made a certain year, the
      #the function simply prints the year.
      try:
        year_mean = mean_sum / data_count
        print(f"{year}: {year_mean} ({pos_sum}/{neu_sum}/{neg_sum})")
      except ZeroDivisionError:
        print(f"-{year}-")
    pass




In [ ]:
#The following code takes three folders with parliament speeches including different location and store their sentiment information.
test_1 = Sum("example//directory//", "Arvika")
test_2 = Sum("directory//example//", "Blekinge")
test_3 = Sum("direx//ectomple", "Cypern")

test_1.store()
test_2.store()
test_3.store()

In [ ]:
#After having moved the stored files into a commmon folder, the followeing code will produce a table with sentiment data from
#the three locations taken together.
display_test = Group("example//directory//")
display_test.table()